# Notes

The `pem` files are in `ASN.1` format with `DER` encoding which is basiacally `tag,length,value`

This is the offical [RSA](https://www.rfc-editor.org/rfc/rfc8017) Private Key sequence 

```
      RSAPrivateKey ::= SEQUENCE {
          version           Version,
          modulus           INTEGER,  -- n
          publicExponent    INTEGER,  -- e
          privateExponent   INTEGER,  -- d
          prime1            INTEGER,  -- p
          prime2            INTEGER,  -- q
          exponent1         INTEGER,  -- d mod (p-1)
          exponent2         INTEGER,  -- d mod (q-1)
          coefficient       INTEGER,  -- (inverse of q) mod p
          otherPrimeInfos   OtherPrimeInfos OPTIONAL
      }
```

```
   o  version is the version number, for compatibility with future
      revisions of this document.  It SHALL be 0 for this version of the
      document, unless multi-prime is used; in which case, it SHALL be
      1.

            Version ::= INTEGER { two-prime(0), multi(1) }
               (CONSTRAINED BY
               {-- version must be multi if otherPrimeInfos present --})
   o  modulus is the RSA modulus n.
   o  publicExponent is the RSA public exponent e.
   o  privateExponent is the RSA private exponent d.
   o  prime1 is the prime factor p of n.
   o  prime2 is the prime factor q of n.
   o  exponent1 is d mod (p - 1).
   o  exponent2 is d mod (q - 1).
   o  coefficient is the CRT coefficient q^(-1) mod p.
```

However, there are two differences between RSA [Formats](https://stackoverflow.com/questions/20065304/differences-between-begin-rsa-private-key-and-begin-private-key):

- `BEGIN PRIVATE KEY` -> PKCS#8 just base64 encoded 
- `BEGIN RSA PRIVATE KEY` -> PKCS#1 which maps to the above DER ASN.1 sequence

In [1]:
from base64 import b64decode

from asn1 import Decoder,Encoder
from Crypto.Util.number import long_to_bytes,bytes_to_long

In [2]:
with open("key.pem","rb") as f:
    # Step 1: Clean and decode base64
    b64 = b''.join(line for line in f.readlines() if not line.startswith(b'---'))
    data = b64decode(b64 + b"===")


In [3]:
fields = [
"version",
"modulus",
"publicExponent",
"privateExponent",
"prime1",
"prime2",
"exponent1",
"exponent2",
"coefficient",
"otherPrimeInfos",
]

In [4]:
# Apparently, the VERION field is not part of the standard

decoder = Decoder()
decoder.start(data)

tag = decoder.peek()
print(tag)
decoder.enter()

rsa = dict()

for field in fields:
    try:
        tag,value = decoder.read()
        rsa[field] = value
        print(value.bit_length())
    except Exception as e:
        print(e)
        break

rsa

Tag(nr=<Numbers.Sequence: 0x10>, typ=<Types.Constructed: 0x20>, cls=<Classes.Universal: 0x00>)
0
4096
17
4091
'NoneType' object has no attribute 'bit_length'


{'version': 0,
 'modulus': 6416469299744099775363894718711925033507924673517652951120791163569587218042876236341402642830163637176337251699852299303256503120343931836214809980020973784303471296245281117640736376578347300506000325049486729917241153262516045288360887608969932360756279150134644997665322211818194295457661751742757924884495260405385051737596513134600242162903302564760336532401113057723762969093761156344875589308961011730355946978481803564938789354209261121014052633490991037136743376153503528129540665538252518481131566474244868415451120347376615633025017916504792941763752900305724967249765207157953723075572043933475154451009699167301210580433206710319359806177360759510270947557667797210072040895424010053682322740165875517150249693837140856616792836953606367924364783529022459498130520240954450298232208743880975540156914239182041646933848986813168984684163205491881548566106729179310242687838744716643439443300162690928175393176294689880138026121241782106710744025424303885048980

In [58]:
import Crypto.PublicKey.RSA as RSA
from Crypto.Cipher import PKCS1_v1_5

In [60]:
key = RSA.construct(
    (rsa["modulus"],rsa["publicExponent"],rsa["privateExponent"])
)

with open("encrypted.txt","rb") as f:
    print(
        PKCS1_v1_5.new(key).decrypt(f.read(),None)
    )

bytearray(b'CTF{learning_the_der_encoding_helps}')
